# OpenFold3 Length Benchmark

Notebook UI for the experiment "protein length vs OpenFold3 RMSD".

Workflow:
1. Fill in `pdb_ids_text` and runtime settings.
2. Run the preview cell to inspect detected protein composition.
3. Run the benchmark cell to launch OpenFold3 and RMSD evaluation.
4. Inspect the result table, summary, and generated SVG plots.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from openfold3_length_benchmark import (
    RuntimeConfig,
    build_notebook_controls,
    preview_entries,
    run_length_benchmark,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)


## Runtime Setup

Adjust these paths if your OpenFold3 environment or MSA cache lives elsewhere.


In [ ]:
OPENFOLD_REPO_DIR = PROJECT_ROOT
NOTEBOOK_HELPERS_DIR = PROJECT_ROOT

runtime = RuntimeConfig(
    project_dir=OPENFOLD_REPO_DIR,
    openfold_repo_dir=OPENFOLD_REPO_DIR,
    openfold_prefix=OPENFOLD_REPO_DIR / ".conda",
    results_dir=PROJECT_ROOT / "openfold3_length_benchmark" / "runs" / "_scratch",
    msa_cache_dir=NOTEBOOK_HELPERS_DIR / ".runtime" / "of3_colabfold_msas",
    triton_cache_dir=PROJECT_ROOT / "openfold3_length_benchmark" / ".runtime" / "triton_cache",
    fixed_msa_tmp_dir=PROJECT_ROOT / "openfold3_length_benchmark" / ".runtime" / "of3_colabfold_msas",
    use_fused_attention=False,
    use_deepspeed=False,
)

print("OPENFOLD_REPO_DIR:", OPENFOLD_REPO_DIR)
runtime


## User Input

`templates` are intentionally disabled inside the benchmark runner.


In [ ]:
pdb_ids_text = """1L2Y
1VII
1CRN
1PGA
2QMT
1BDD
2K39
1UBQ
2PTL
1R6J
2CI2
1ZAA
1TEN
1QYS
1BKR
1AKI
1MBA
1STN
1RX2
3LZM
5U32
4ZIB
1FNF
1I5P
6JDQ
5Y36
6O0Y
"""

# pdb_ids_text = """1GT0
# 4AKE
# 7DUN
# 1TIM
# 1AO6
# 4K2C
# 6ZSL
# 7LBY
# 4CMP
# 4OO8
# """

atom_set = "ca"
use_msa_server = True
num_diffusion_samples = 1
num_model_seeds = 1
runner_yaml = None
output_root = PROJECT_ROOT / "openfold3_length_benchmark" / "runs"
max_entries = None

NOTEBOOK_STATE = {}


## Optional Widget Controls

If `ipywidgets` is installed in the notebook kernel, these buttons provide quick preview/run/refresh actions.
If not, the cell returns a fallback message and the normal cells below remain fully runnable.


In [ ]:
controls = build_notebook_controls(
    runtime=runtime,
    config_getter=lambda: {
        "pdb_ids_text": pdb_ids_text,
        "atom_set": atom_set,
        "use_msa_server": use_msa_server,
        "num_diffusion_samples": num_diffusion_samples,
        "num_model_seeds": num_model_seeds,
        "runner_yaml": runner_yaml,
        "output_root": output_root,
        "max_entries": max_entries,
    },
    state=NOTEBOOK_STATE,
)
controls


## Preview Entry Composition


In [ ]:
preview_df = preview_entries(
    pdb_ids_text,
    max_entries=max_entries,
)
NOTEBOOK_STATE["preview_df"] = preview_df

display(preview_df)


## Run Benchmark


In [ ]:
result = run_length_benchmark(
    runtime=runtime,
    pdb_ids=pdb_ids_text,
    atom_set=atom_set,
    use_msa_server=use_msa_server,
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    runner_yaml=runner_yaml,
    output_root=output_root,
    max_entries=max_entries,
)
NOTEBOOK_STATE["result"] = result

print("Run root:", result.run_root)
print("Summary path:", result.summary_path)
display(result.results_df)


## Failures And Summary


In [ ]:
display(result.failures_df)
result.summary


## Generated Plots


In [ ]:
for label, plot_path in result.plot_paths.items():
    print(label, "->", plot_path)
    try:
        from IPython.display import SVG, display as ipy_display
        ipy_display(SVG(filename=str(plot_path)))
    except Exception as exc:
        print("SVG inline display is unavailable:", exc)
        print(plot_path.read_text(encoding="utf-8")[:400])
